# Anomaly detection 1/3 — K-means distance scoring

TinyML course, Module 8. Adapted from the original course notebook (`Anomaly Kmeans.ipynb`).

**Idea:** fit K cluster centroids to *normal* data only. The anomaly score of a new sample is its distance to the nearest centroid — far from every centroid means "never seen anything like this".

This is exactly what the Edge Impulse *K-means Anomaly Detection* block does on-device.

Plan:
1. Warm-up on synthetic 2-D data (easy to plot)
2. The same recipe on **your fan features**, with one fault class held out as the "unknown fault"
3. Where K-means breaks → hand-off to the GMM notebook

## Part 1 — Synthetic warm-up

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

rng = np.random.default_rng(42)

# Normal data: three blobs. Anomalies: a small cloud around (10, 10).
X_norm, _ = make_blobs(n_samples=950, n_features=2, centers=3, random_state=42)
X_anom = np.array([[10.0, 10.0]]) + rng.normal(size=(50, 2)) * 2.0
X = np.vstack([X_norm, X_anom])

plt.figure(figsize=(7, 4))
plt.plot(X_norm[:, 0], X_norm[:, 1], 'k.', ms=3, label='normal')
plt.plot(X_anom[:, 0], X_anom[:, 1], 'r.', ms=5, label='anomalies')
plt.legend(); plt.title('Synthetic data'); plt.show()

In [ ]:
# Fit K-means on the NORMAL data only, then score every point by the
# distance to its nearest centroid.
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_norm)

def distance_score(model, X):
    """Distance to the nearest cluster centroid, per sample."""
    d = np.linalg.norm(X[:, None, :] - model.cluster_centers_[None, :, :], axis=2)
    return d.min(axis=1)

scores_norm = distance_score(kmeans, X_norm)
scores_all = distance_score(kmeans, X)

plt.figure(figsize=(7, 3))
plt.hist(scores_norm, bins=30, alpha=0.7)
plt.title('Distance scores of NORMAL training data')
plt.xlabel('distance to nearest centroid'); plt.show()

In [ ]:
# Threshold: right tail of the normal-score distribution, e.g. 98th percentile.
# (Then tune by hand: lower threshold = more sensitivity = more false alarms.)
threshold = np.percentile(scores_norm, 98)
print(f'threshold = {threshold:.3f}')

anomalies = X[scores_all > threshold]
plt.figure(figsize=(7, 4))
plt.scatter(X[:, 0], X[:, 1], s=8, c='lightgray')
plt.scatter(anomalies[:, 0], anomalies[:, 1], s=12, c='purple', label='flagged')
plt.scatter(*kmeans.cluster_centers_.T, marker='x', s=100, c='k', label='centroids')
plt.legend(); plt.title(f'Flagged as anomalous (score > {threshold:.2f})'); plt.show()

## Part 2 — Your fan data

Now the real thing. We load the fan recordings from Module 7, compute the same 13 features (std, MAD, kurtosis, RMS per axis + mean resultant), and:

- **train** K-means on `normal` windows only
- **hold out one fault class entirely** — set `HELD_OUT` below to the class you're pretending nobody ever recorded (worn fan / `scrape`)
- check that the detector flags it anyway

Because K-means uses Euclidean distance, features must be **scaled** — a feature with a big numeric range would otherwise dominate the distance. (Note the contrast with the Random Forest, which needed no scaling.)

In [ ]:
import glob, os, sys

# Reuse the Module 7 loader + feature code so Python == C == EI stays one story.
sys.path.insert(0, os.path.abspath('../../module7-models/rf-features'))
from train_rf import load_windows, WINDOW, HOP   # noqa: E402

DATA_DIR = '../../module7-models/rf-features/data/raw'
HELD_OUT = 'scrape'   # <-- your held-out 'unknown' fault class

X_raw, X_feat, y, n_files = load_windows(DATA_DIR)
print(f'{n_files} files, {len(y)} windows, classes: {sorted(set(y))}')

feat_names = ([f'std_{a}' for a in 'xyz'] + [f'mad_{a}' for a in 'xyz'] +
              [f'kurt_{a}' for a in 'xyz'] + [f'rms_{a}' for a in 'xyz'] + ['res_mean'])

In [ ]:
from sklearn.preprocessing import StandardScaler

# Split: 70 % of the NORMAL windows for training; everything else is test.
is_normal = (y == 'normal')
idx_normal = np.where(is_normal)[0]
rng = np.random.default_rng(0)
rng.shuffle(idx_normal)
n_train = int(0.7 * len(idx_normal))
train_idx = idx_normal[:n_train]

scaler = StandardScaler().fit(X_feat[train_idx])
Z = scaler.transform(X_feat)

km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z[train_idx])
scores = distance_score(km, Z)
threshold = np.percentile(scores[train_idx], 98)
print(f'threshold (98th pct of normal training scores) = {threshold:.3f}')

In [ ]:
# Score distribution per machine state. The held-out class was never used for
# anything until this cell.
import pandas as pd

test_mask = np.ones(len(y), dtype=bool)
test_mask[train_idx] = False
df = pd.DataFrame({'state': y[test_mask], 'score': scores[test_mask]})

ax = df.boxplot(column='score', by='state', figsize=(8, 4))
ax.axhline(threshold, color='r', ls='--', label='threshold')
plt.suptitle(''); plt.title('K-means anomaly score per fan state'); plt.legend(); plt.show()

print('\nDetection summary (fraction of windows flagged as anomalous):')
for state, grp in df.groupby('state'):
    tag = ' <- held out!' if state == HELD_OUT else ''
    print(f"  {state:12s} {np.mean(grp['score'] > threshold):5.1%}{tag}")

**Read the summary like this:**
- `normal` should be flagged ~2 % of the time (that *is* the 98th-percentile threshold — the built-in false-alarm rate)
- every fault state — **including the held-out one** — should be flagged at a high rate

Questions:
1. Move the threshold to the 95th and 99.5th percentile. What happens to false alarms vs missed detections?
2. Change `n_clusters` to 1 and to 8. Why does a very high K make the detector *blinder*? (Hint: enough centroids will eventually sit near anything.)
3. Which fault state is hardest to detect? Compare with the confusion matrix from Module 7 — is it the same troublemaker?

## Part 3 — Where K-means breaks

In [ ]:
# Elliptic, correlated clusters: Euclidean distance draws spherical boundaries
# and gets the borders visibly wrong. (Same demo as the original notebook.)
X1, y1 = make_blobs(n_samples=1000, centers=((4, -4), (0, 0)), random_state=42)
X1 = X1.dot(np.array([[0.374, 0.95], [0.732, 0.598]]))
X2, _ = make_blobs(n_samples=250, centers=1, random_state=42)
X2 = X2 + [6, -8]
Xe = np.r_[X1, X2]

km_e = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Xe)

# Voronoi-style decision regions
mins, maxs = Xe.min(0) - 0.5, Xe.max(0) + 0.5
xx, yy = np.meshgrid(np.linspace(mins[0], maxs[0], 400), np.linspace(mins[1], maxs[1], 400))
Zg = km_e.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.figure(figsize=(8, 4))
plt.contourf(xx, yy, Zg, cmap='Pastel2')
plt.contour(xx, yy, Zg, colors='k', linewidths=0.5)
plt.plot(Xe[:, 0], Xe[:, 1], 'k.', ms=2)
plt.scatter(*km_e.cluster_centers_.T, marker='x', s=100, c='r')
plt.title('K-means on elliptic clusters: boundaries cut through the data'); plt.show()

K-means is not really good at this — now what…?

Give the clusters a *shape*: means **and covariances**. That's a Gaussian Mixture Model.

**→ continue with `02_gmm_anomaly.ipynb`** (it reuses the fan features you just computed).